# 04. Output Parser

LLM의 출력을 원하는 형식(문자열, JSON, 객체 등)으로 파싱합니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. StrOutputParser - 문자열 추출

`AIMessage` 객체에서 `.content` 문자열만 추출합니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}을 한 문장으로 설명해줘")
])

parser = StrOutputParser()

messages = prompt.invoke({"topic": "LangChain"})
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(response))
print(result)


<class 'langchain_core.messages.ai.AIMessage'>
LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.


## 2. JsonOutputParser - JSON 파싱

LLM이 JSON을 반환하도록 유도하고 딕셔너리로 파싱합니다.

In [5]:
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("human", "파이썬 언어에 대한 정보를 name, year, creator 키를 가진 JSON으로 반환해줘. JSON만 반환해.")
])
parser = JsonOutputParser()

messages = prompt.invoke({})
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(response))
print(result)



<class 'langchain_core.messages.ai.AIMessage'>
{'name': 'Python', 'year': 1991, 'creator': 'Guido van Rossum'}


## 3. PydanticOutputParser - 구조체 파싱

Pydantic 모델을 정의하면 자동으로 포맷 지시사항을 생성하고 파싱합니다.

In [6]:

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    score: str = Field(description="영화 리뷰")
    summary: str = Field(description="한줄평")

parser = PydanticOutputParser(pydantic_object=MovieReview)

print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "영화 제목", "title": "Title", "type": "string"}, "score": {"description": "영화 리뷰", "title": "Score", "type": "string"}, "summary": {"description": "한줄평", "title": "Summary", "type": "string"}}, "required": ["title", "score", "summary"]}
```


In [7]:
영화 인터스텔라에 대한 평가 부탁해

SyntaxError: invalid syntax (895751907.py, line 1)

## 4. CommaSeparatedListOutputParser - 리스트 파싱

<class 'list'>
['Django', 'Flask', 'FastAPI', 'Pyramid', 'Tornado']
